In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [3]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]


from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [5]:
import json
json.dumps(documents[0])

'{"content": "# Introduction\\n\\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\\n\\nIn this module, we\'ll build a working Retrieval-Augmented\\nGeneration (RAG) system from scratch, step by step.\\n\\nWe write everything in plain Python. We build a small search index by\\nhand and call the LLM ourselves. I want you to see every piece first.\\nThat way you know what a framework does for you before you reach for\\none.\\n\\nPlaces where you can find me:\\n\\n- [My substack](https://alexeyondata.substack.com/)\\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\\n- [X](https://x.com/Al_Grigor)\\n\\n## LLMs\\n\\nAn LLM (Large Language Model) is a neural network trained on massive\\namounts of text. Given a prompt, it generates a continuation - a\\nplausible next piece of text.\\n\\nThink of your phone. When you type \\"how are\\" in WhatsApp, it suggests\\n\\"you\\" as the next word. \\"How are you\\" is the most common

In [4]:
from evaluation_utils import llm_structured_retry

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["filename"]
        })

    return results, usage

In [12]:
generate_ground_truth(documents[0])

([{'question': 'What does a retrieval-augmented generation system do that a plain LLM can’t do well on its own?',
   'document': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'Why does the course treat the language model like a black box instead of explaining how it works internally?',
   'document': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'What are the main limits of LLMs that RAG is supposed to fix?',
   'document': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'What kind of example project will be built in this module to show RAG in practice?',
   'document': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'How is the module split up, and what changes in the second part compared with the first part?',
   'document': '01-agentic-rag/lessons/01-intro.md'}],
 ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=114, output_tokens_details=OutputTokensDetails(reasoning_tokens=0),

In [13]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [14]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/72 [00:00<?, ?it/s]

## Q1

In [24]:
[res[1].input_tokens for res in results[:3]]

[1020, 1286, 1753]

## Q2

In [5]:
import pandas as pd
ground_truth = pd.read_csv("ground-truth.csv")

In [6]:
ground_truth.head(3)

,question,filename
0,What exactly is a retrieval-augmented generati...,01-agentic-rag/lessons/01-intro.md
1,Why does this course build the RAG project in ...,01-agentic-rag/lessons/01-intro.md
2,What are the main weaknesses of large language...,01-agentic-rag/lessons/01-intro.md


In [11]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(documents)


In [45]:
def text_search(query, num_results=5): 
    return index.search(
        query,
        num_results=num_results
    )

In [7]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Number of chunks: {len(chunks)}")

Number of chunks: 295


In [8]:
batch = [chunk['content'] for chunk in chunks]

from embedder import Embedder

2026-07-09 21:51:17.607123184 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [9]:
model = Embedder()
X = model.encode_batch(batch)

In [10]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['filename'])
vindex.fit(X, chunks)

In [43]:
def vector_search(query, num_results=5):
    q_vector = model.encode(query)
    return vindex.search(
        q_vector,
        num_results=num_results
    )

In [66]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [59]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k, num_results=10)

## Q2 text search

In [24]:
q = ground_truth.iloc[0]["question"]

In [27]:
text_search(q, num_results=5)

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

## Q3 vector search

In [28]:
vector_search(q, num_results=5)

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

## Q4 eval text search

In [56]:
from tqdm.auto import tqdm

def compute_relevance(gtrec, search_function):
    doc_fn = gtrec["filename"]
    results = search_function(query=gtrec["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_fn))

    return relevance

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for gtr in tqdm(ground_truth):
        relevance = compute_relevance(gtr, search_function)
        relevance_total.append(relevance)

    return relevance_total


In [61]:
ground_truth_lst = ground_truth.to_dict(orient="records")

In [46]:
relevance_total_text = compute_relevance_total(ground_truth_lst, text_search)

  0%|          | 0/360 [00:00<?, ?it/s]

In [48]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)

def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [49]:
evaluate(ground_truth_lst, text_search) 

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7555555555555555, 'mrr': 0.6037962962962965}

## Q5 eval vector search

In [58]:
evaluate(ground_truth_lst, vector_search) 

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

## Q6 hybrid search

In [67]:
for param_k in [1, 50, 100, 200]:
    print(f"Evaluating hybrid search with k={param_k}...")
    def hybrid_search_with_k(query):
        return hybrid_search(query=query, k=param_k)
    results = evaluate(ground_truth_lst, hybrid_search_with_k)
    print(f"Results for k={param_k}: {results}")

Evaluating hybrid search with k=1...


  0%|          | 0/360 [00:00<?, ?it/s]

Results for k=1: {'hit_rate': 0.9166666666666666, 'mrr': 0.6678406084656087}
Evaluating hybrid search with k=50...


  0%|          | 0/360 [00:00<?, ?it/s]

Results for k=50: {'hit_rate': 0.9166666666666666, 'mrr': 0.5901598324514991}
Evaluating hybrid search with k=100...


  0%|          | 0/360 [00:00<?, ?it/s]

Results for k=100: {'hit_rate': 0.9166666666666666, 'mrr': 0.5901598324514991}
Evaluating hybrid search with k=200...


  0%|          | 0/360 [00:00<?, ?it/s]

Results for k=200: {'hit_rate': 0.9166666666666666, 'mrr': 0.5901598324514991}
